In [1]:
!pip install pycryptodome scikit-image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 19.9 MB/s  0:00:00


In [3]:
import os
import hashlib
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import csv
import math
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
from Crypto.Random import get_random_bytes
from Crypto.Protocol.KDF import PBKDF2
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# ==========================================
# 1. Вспомогательные функции
# ==========================================
def load_image(path):
    return np.array(Image.open(path).convert('RGB'), dtype=np.uint8)

def save_image(data, path):
    # Принудительно сохраняем в PNG, чтобы избежать потерь сжатия
    Image.fromarray(data).save(path, format='PNG')

# ==========================================
# 2. Шифрование и Дешифрование
# ==========================================
def encrypt_message(message, password):
    salt = get_random_bytes(16)
    iv = get_random_bytes(16)
    key = PBKDF2(password, salt, dkLen=32)
    cipher = AES.new(key, AES.MODE_CBC, iv)
    ciphertext = cipher.encrypt(pad(message.encode('utf-8'), AES.block_size))
    # Формат: [длина (4 байта)] + [salt] + [iv] + [ciphertext]
    length_bytes = len(ciphertext).to_bytes(4, 'big')
    return length_bytes + salt + iv + ciphertext

def decrypt_message(enc_data, password):
    length = int.from_bytes(enc_data[:4], 'big')
    salt = enc_data[4:20]
    iv = enc_data[20:36]
    ciphertext = enc_data[36:36+length]
    key = PBKDF2(password, salt, dkLen=32)
    cipher = AES.new(key, AES.MODE_CBC, iv)
    decrypted = unpad(cipher.decrypt(ciphertext), AES.block_size)
    return decrypted.decode('utf-8', errors='ignore')

# ==========================================
# 3. LSB Стеганография (Встраивание и Извлечение)
# ==========================================
def embed_lsb_sequential(cover, payload_bytes):
    stego = cover.copy()
    flat_img = stego.flatten().astype(int)

    # Записываем длину сообщения в первые 4 байта (32 бита)
    length_bits = [int(b) for b in f'{len(payload_bytes):032b}']
    payload_bits = [int(b) for byte in payload_bytes for b in f'{byte:08b}']
    all_bits = length_bits + payload_bits

    if len(all_bits) > len(flat_img):
        raise ValueError("Сообщение слишком большое!")

    for i, bit in enumerate(all_bits):
        flat_img[i] = (flat_img[i] & ~1) | bit

    return flat_img.astype(np.uint8).reshape(cover.shape)

def extract_lsb_sequential(stego):
    flat_img = stego.flatten()

    # Сначала читаем 32 бита длины
    length_bits = [flat_img[i] & 1 for i in range(32)]
    msg_len = int("".join(map(str, length_bits)), 2)

    # Читаем само сообщение
    msg_bits = [flat_img[i] & 1 for i in range(32, 32 + msg_len * 8)]
    payload_bytes = bytes([int("".join(map(str, msg_bits[i:i+8])), 2) for i in range(0, len(msg_bits), 8)])

    return payload_bytes

def embed_lsb_random(cover, payload_bytes, password):
    stego = cover.copy()
    flat_img = stego.flatten().astype(int)

    length_bits = [int(b) for b in f'{len(payload_bytes):032b}']
    payload_bits = [int(b) for byte in payload_bytes for b in f'{byte:08b}']
    all_bits = length_bits + payload_bits

    if len(all_bits) > len(flat_img):
        raise ValueError("Сообщение слишком большое!")

    seed = int(hashlib.sha256(password.encode()).hexdigest(), 16) % (2**32)
    np.random.seed(seed)
    indices = np.random.choice(len(flat_img), size=len(all_bits), replace=False)

    for idx, bit in zip(indices, all_bits):
        flat_img[idx] = (flat_img[idx] & ~1) | bit

    return flat_img.astype(np.uint8).reshape(cover.shape)

def extract_lsb_random(stego, password):
    flat_img = stego.flatten()

    seed = int(hashlib.sha256(password.encode()).hexdigest(), 16) % (2**32)
    np.random.seed(seed)

    # Сначала восстанавливаем индексы для чтения длины
    indices_len = np.random.choice(len(flat_img), size=32, replace=False)
    length_bits = [flat_img[idx] & 1 for idx in indices_len]
    msg_len = int("".join(map(str, length_bits)), 2)

    # Теперь восстанавливаем индексы для чтения самого сообщения
    np.random.seed(seed) # Сбрасываем seed, чтобы последовательность была той же
    indices_msg = np.random.choice(len(flat_img), size=32 + msg_len * 8, replace=False)
    msg_bits = [flat_img[idx] & 1 for idx in indices_msg[32:]]
    payload_bytes = bytes([int("".join(map(str, msg_bits[i:i+8])), 2) for i in range(0, len(msg_bits), 8)])

    return payload_bytes

# ==========================================
# 4. Метрики и Визуализация
# ==========================================
def calculate_metrics(cover, stego):
    p = psnr(cover, stego, data_range=255)
    s = ssim(cover, stego, data_range=255, channel_axis=2)
    diff = np.sum(cover != stego)
    total = cover.size
    changed_ratio = diff / total
    lsb_ones = np.sum(stego.flatten() & 1)
    p_lsb_1 = lsb_ones / total

    flat = stego.flatten()
    counts = np.bincount(flat, minlength=256)
    chi2 = 0.0
    df = 0
    for i in range(0, 256, 2):
        n_even = counts[i]
        n_odd = counts[i+1]
        n_total = n_even + n_odd
        if n_total > 0:
            expected = n_total / 2.0
            chi2 += ((n_even - expected)**2) / expected
            chi2 += ((n_odd - expected)**2) / expected
            df += 1
    if df > 0:
        z = (chi2 - df) / math.sqrt(2 * df)
        p_val = 0.5 * math.erfc(z / math.sqrt(2))
    else:
        p_val = 1.0
    return p, s, changed_ratio, p_lsb_1, chi2, p_val

def plot_visuals(cover, stego, img_name, method_name):
    # Карта разности
    diff = np.abs(cover.astype(int) - stego.astype(int))
    diff_map = np.zeros_like(cover)
    mask = np.sum(diff, axis=2) > 0
    diff_map[mask] = [255, 0, 0]
    save_image(diff_map, f'StegoCrypto/results/difference_{img_name}_{method_name}.png')

    # Гистограмма
    plt.figure(figsize=(10, 5))
    plt.hist(cover.flatten(), bins=256, alpha=0.5, label='Cover', color='blue')
    plt.hist(stego.flatten(), bins=256, alpha=0.5, label='Stego', color='red')
    plt.title(f'Histogram: {img_name} ({method_name})')
    plt.legend()
    plt.savefig(f'StegoCrypto/results/histogram_{img_name}_{method_name}.png')
    plt.close()

# ==========================================
# 5. Основной запуск
# ==========================================
def run_stego_experiment(img_path, img_name, message, password, payload_percent):
    print(f"\n{'='*50}\nАнализ: {img_name} | Payload: {payload_percent}%\n{'='*50}")

    os.makedirs('StegoCrypto/imgs', exist_ok=True)
    os.makedirs('StegoCrypto/results', exist_ok=True)

    cover = load_image(img_path)
    # Сохраняем cover в PNG
    cover_png_path = f'StegoCrypto/imgs/cover_{img_name}.png'
    save_image(cover, cover_png_path)

    capacity_bits = cover.size
    target_bits = int(capacity_bits * (payload_percent / 100.0))
    target_bytes = target_bits // 8

    base_message = message.encode('utf-8')
    if len(base_message) < target_bytes:
        padding_needed = target_bytes - len(base_message)
        message_bytes = base_message + b'A' * padding_needed
    else:
        message_bytes = base_message[:target_bytes]

    print(f"Размер встроенного сообщения: {len(message_bytes)} байт (заполнение {payload_percent}%)")

    encrypted_data = encrypt_message(message_bytes.decode('utf-8', errors='ignore'), password)

    stego_plain = embed_lsb_sequential(cover, message_bytes)
    stego_enc = embed_lsb_sequential(cover, encrypted_data)
    stego_rand = embed_lsb_random(cover, encrypted_data, password)

    prefix = f"{img_name}_p{payload_percent}"
    save_image(stego_plain, f'StegoCrypto/imgs/stego_plain_{prefix}.png')
    save_image(stego_enc, f'StegoCrypto/imgs/stego_encrypted_{prefix}.png')
    save_image(stego_rand, f'StegoCrypto/imgs/stego_random_{prefix}.png')

    # ==========================================
    # ПРОВЕРКА ИЗВЛЕЧЕНИЯ
    # ==========================================
    print("\n--- Проверка извлечения данных ---")

    # 1. Извлечение Plain
    extracted_plain = extract_lsb_sequential(stego_plain)
    print(f"Plain LSB восст: {extracted_plain == message_bytes}")

    # 2. Извлечение Encrypted (последовательный)
    extracted_enc_bytes = extract_lsb_sequential(stego_enc)
    decrypted_enc = decrypt_message(extracted_enc_bytes, password)
    print(f"Encrypted LSB восст: {decrypted_enc.encode('utf-8') == message_bytes}")

    # 3. Извлечение Encrypted (случайный)
    extracted_rand_bytes = extract_lsb_random(stego_rand, password)
    decrypted_rand = decrypt_message(extracted_rand_bytes, password)
    print(f"Random LSB восст: {decrypted_rand.encode('utf-8') == message_bytes}")

    methods = [("Plain LSB", stego_plain), ("Encrypted LSB", stego_enc), ("Random LSB", stego_rand)]
    metrics = []

    print(f"\n{'Метод':<15} | {'PSNR':<8} | {'SSIM':<8} | {'Changed':<8} | {'P(LSB=1)':<10} | {'chi2':<8} | {'p-value':<8}")
    print("-" * 85)

    for name, stego in methods:
        p, s, cr, p1, c2, pv = calculate_metrics(cover, stego)
        metrics.append([name, p, s, cr, p1, c2, pv])
        print(f"{name:<15} | {p:.2f}     | {s:.4f}   | {cr:.4f}   | {p1:.4f}     | {c2:.2f}   | {pv:.4f}")

        if name == "Encrypted LSB":
            plot_visuals(cover, stego, img_name, prefix)

    with open(f'StegoCrypto/results/{prefix}_metrics.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Method', 'PSNR', 'SSIM', 'Changed_ratio', 'P_LSB_1', 'Chi2', 'P_value'])
        writer.writerows(metrics)

# ==========================================
# ЗАПУСК (используем PNG файлы, уровни 1% и 5%)
# ==========================================
MESSAGE = "Hello, this is a secret message for the lab!"
PASSWORD = "SuperSecretPassword123"

# Убедитесь, что эти файлы загружены в Colab
run_stego_experiment('gradient.png', 'gradient.png', MESSAGE, PASSWORD, 1)
run_stego_experiment('gradient.png', 'gradient.png', MESSAGE, PASSWORD, 5)
run_stego_experiment('checkerboard.png', 'checkerboard.png', MESSAGE, PASSWORD, 1)
run_stego_experiment('checkerboard.png', 'checkerboard.png', MESSAGE, PASSWORD, 5)

print("\nГотово! Все картинки, графики и таблицы сохранены в StegoCrypto/")


Анализ: gradient.png | Payload: 1%
Размер встроенного сообщения: 983 байт (заполнение 1%)


/tmp/ipykernel_817/235270764.py:31: DeprecationWarning: PBKDF2 without 'count' (1000 iterations) and 'hmac_hash_module' (HMAC-SHA1) uses weak defaults, which are deprecated and will be removed: pass them explicitly (for instance, count=600000, hmac_hash_module=SHA256)
  key = PBKDF2(password, salt, dkLen=32)



--- Проверка извлечения данных ---
Plain LSB восст: True
Encrypted LSB восст: True
Random LSB восст: True

Метод           | PSNR     | SSIM     | Changed  | P(LSB=1)   | chi2     | p-value 
-------------------------------------------------------------------------------------


/tmp/ipykernel_817/235270764.py:43: DeprecationWarning: PBKDF2 without 'count' (1000 iterations) and 'hmac_hash_module' (HMAC-SHA1) uses weak defaults, which are deprecated and will be removed: pass them explicitly (for instance, count=600000, hmac_hash_module=SHA256)
  key = PBKDF2(password, salt, dkLen=32)


Plain LSB       | 71.50     | 1.0000   | 0.0046   | 0.4898     | 8616.81   | 0.0000
Encrypted LSB   | 70.97     | 1.0000   | 0.0052   | 0.4923     | 8639.08   | 0.0000
Random LSB      | 70.98     | 0.9999   | 0.0052   | 0.4917     | 8527.72   | 0.0000

Анализ: gradient.png | Payload: 5%
Размер встроенного сообщения: 4915 байт (заполнение 5%)

--- Проверка извлечения данных ---
Plain LSB восст: True
Encrypted LSB восст: True
Random LSB восст: True

Метод           | PSNR     | SSIM     | Changed  | P(LSB=1)   | chi2     | p-value 
-------------------------------------------------------------------------------------
Plain LSB       | 64.35     | 0.9997   | 0.0239   | 0.4804     | 10509.65   | 0.0000
Encrypted LSB   | 64.09     | 0.9996   | 0.0253   | 0.4928     | 8148.54   | 0.0000
Random LSB      | 64.11     | 0.9996   | 0.0252   | 0.4919     | 7837.78   | 0.0000

Анализ: checkerboard.png | Payload: 1%
Размер встроенного сообщения: 983 байт (заполнение 1%)

--- Проверка извлечения данны